# 01 - Image classifier (binary head + multilabel head)

One backbone, two heads:
- **Binary head** -> `p_abn`: probability the image is abnormal (labels agree 100%).
- **Multilabel head** -> `p_c0..p_c13`: probability each class is present, trained on soft labels (fraction of radiologists).

**Inputs:** output of `00_prep` + `vbd_utils`. **GPU required.**

**Outputs:** `oof_cls_fold{k}.csv` (validation predictions), `test_cls_fold{k}.csv`, `cls_fold{k}.pt`.

**Workflow:** set `FOLDS = [0]`, check the report at the bottom, then run the remaining folds (one or more per session). The report always uses every OOF file it can find, so results from earlier sessions are included when their outputs are attached.

In [ ]:
# Import the shared utilities (upload vbd_utils.py as a Kaggle Dataset and attach it)
import os, sys, glob, warnings
def _locate(name, root='/kaggle/input'):
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('png1024', 'images', 'labels', 'train', 'test')]
        if name in files:
            return cur
    raise FileNotFoundError(f'{name} not found - attach the vbd-utils dataset')
sys.path.insert(0, _locate('vbd_utils.py'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import vbd_utils as U
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
OUT = '/kaggle/working'

In [ ]:
# ---------------- configuration ----------------
FOLDS = [0]                  # e.g. [0] first, then [1, 2], [3, 4]
BACKBONE = 'efficientnet_b3' # any timm model name, e.g. 'convnext_tiny'
IMG = 768                    # 1024 if memory allows
EPOCHS = 15
BATCH = 8                    # 16 runs out of memory on a T4 at 768 px
LR = 3e-4
LAMBDA_ML = 1.0              # weight of the multilabel loss
NUM_WORKERS = 4
SEED = 42

In [ ]:
import cv2, timm, torch, torch.nn as nn
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda'
PNG_DIR = U.find_dir('png1024')
meta = pd.read_csv(U.find_file('meta.csv'))
labels = pd.read_csv(U.find_file('cls_labels.csv'), index_col='image_id')
folds = pd.read_csv(U.find_file('folds.csv'), index_col=0).fold
targets = pd.read_csv(U.find_file('targets_fused.csv'))
test_ids = meta.loc[meta.split == 'test', 'image_id'].tolist()
CCOLS = [f'c{c}' for c in range(14)]
print(len(labels), 'train images |', len(test_ids), 'test images')

## Dataset and model

In [ ]:
aug = A.Compose([
    A.Affine(translate_percent=(-0.1, 0.1), scale=(0.8, 1.2), rotate=(-5, 5), p=0.7),   # no flips (anatomy)
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.RandomGamma(p=0.3)])
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

class ClsDS(Dataset):
    def __init__(self, ids, labels=None, train=False):
        self.ids, self.labels, self.train = list(ids), labels, train
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        iid = self.ids[i]
        img = cv2.resize(cv2.imread(f'{PNG_DIR}/{iid}.png', 0), (IMG, IMG), interpolation=cv2.INTER_AREA)
        if self.train:
            img = aug(image=img)['image']
        x = (np.repeat(img[..., None], 3, 2) / 255.0 - MEAN) / STD
        x = torch.tensor(x.transpose(2, 0, 1), dtype=torch.float32)
        if self.labels is None:
            return x
        y = self.labels.loc[iid]
        return x, torch.tensor(y.abnormal, dtype=torch.float32), torch.tensor(y[CCOLS].values.astype(np.float32))

class TwoHeadNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0)
        f = self.backbone.num_features
        self.bin_head, self.ml_head = nn.Linear(f, 1), nn.Linear(f, 14)
    def forward(self, x):
        z = self.backbone(x)
        return self.bin_head(z).squeeze(1), self.ml_head(z)

bce = nn.BCEWithLogitsLoss()     # works with soft targets

@torch.no_grad()
def predict(model, ids):
    model.eval()
    dl = DataLoader(ClsDS(ids), batch_size=BATCH * 2, num_workers=NUM_WORKERS)
    pb, pm = [], []
    for x in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            lb, lm = model(x.to(device))
        pb.append(torch.sigmoid(lb.float()).cpu().numpy()); pm.append(torch.sigmoid(lm.float()).cpu().numpy())
    df = pd.DataFrame(np.concatenate(pm), columns=[f'p_c{c}' for c in range(14)])
    df.insert(0, 'p_abn', np.concatenate(pb)); df.insert(0, 'image_id', list(ids))
    return df

def val_scores(pred, ids):
    y = labels.loc[ids]
    auc_bin = roc_auc_score(y.abnormal, pred.p_abn)
    aucs = [roc_auc_score(y[f'c{c}'] > 0, pred[f'p_c{c}']) for c in range(14)]
    return auc_bin, float(np.mean(aucs))

## Training

In [ ]:
def train_fold(k):
    tr_ids, va_ids = folds.index[folds != k], folds.index[folds == k]
    dl = DataLoader(ClsDS(tr_ids, labels, train=True), batch_size=BATCH, shuffle=True,
                    num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
    model = TwoHeadNet().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda')
    hist, best = [], -1
    for ep in range(EPOCHS):
        model.train(); losses = []
        for x, yb, ys in tqdm(dl, desc=f'fold {k} epoch {ep}', leave=False):
            x, yb, ys = x.to(device), yb.to(device), ys.to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                lb, lm = model(x)
                loss = bce(lb.float(), yb) + LAMBDA_ML * bce(lm.float(), ys)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        pred = predict(model, va_ids)
        auc_bin, auc_ml = val_scores(pred, va_ids)
        score = (auc_bin + auc_ml) / 2
        hist.append(dict(epoch=ep, loss=np.mean(losses), auc_bin=auc_bin, auc_ml_macro=auc_ml))
        print(f'fold {k} ep {ep}: loss {np.mean(losses):.4f} | AUC binary {auc_bin:.4f} | AUC multilabel {auc_ml:.4f}')
        if score > best:
            best = score
            torch.save(model.state_dict(), f'{OUT}/cls_fold{k}.pt')
    hist = pd.DataFrame(hist)
    hist.to_csv(f'{OUT}/cls_history_fold{k}.csv', index=False)
    return hist

for k in FOLDS:
    hist = train_fold(k)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    hist.plot(x='epoch', y='loss', ax=ax[0], title=f'fold {k}: training loss')
    hist.plot(x='epoch', y=['auc_bin', 'auc_ml_macro'], ax=ax[1], title=f'fold {k}: validation AUC')
    plt.tight_layout(); plt.show()

    model = TwoHeadNet().to(device)
    model.load_state_dict(torch.load(f'{OUT}/cls_fold{k}.pt', map_location=device))
    predict(model, folds.index[folds == k]).to_csv(f'{OUT}/oof_cls_fold{k}.csv', index=False)
    predict(model, test_ids).to_csv(f'{OUT}/test_cls_fold{k}.csv', index=False)
    del model; torch.cuda.empty_cache()

## Evaluation report
Uses every `oof_cls_fold*.csv` available (this session + attached outputs of earlier sessions).

In [ ]:
oof = U.load_concat('oof_cls_fold*.csv').drop_duplicates('image_id').set_index('image_id')
y = labels.loc[oof.index]
print('images evaluated:', len(oof), '| folds:', sorted(folds.loc[oof.index].unique()))

### Binary head
ROC and PR curves, score distributions, and sensitivity / specificity / confusion matrix at several thresholds:
Youden's J (best balance), 0.5, and the highest thresholds that still reach 95% and 99% sensitivity.

In [ ]:
bin_table = U.binary_report(y.abnormal, oof.p_abn)
display(bin_table)
print('AUC per fold:', {k: round(roc_auc_score(y.abnormal[folds.loc[oof.index] == k], oof.p_abn[folds.loc[oof.index] == k]), 4)
                        for k in sorted(folds.loc[oof.index].unique())})

### Multilabel head
- `AUC_any`: positive = at least one radiologist marked the class.
- `AUC_agreed`: positive = at least two radiologists.
- The right chart checks the reason we added this head: the predicted probability should **rise with radiologist agreement** (0 -> 1/3 -> 2/3 -> 1).

In [ ]:
ml_table, ml_votes = U.multilabel_report(y[CCOLS], oof[[f'p_c{c}' for c in range(14)]])
display(ml_table); display(ml_votes)

### Error analysis: the most confident mistakes of the binary head

In [ ]:
def show_ids(ids, title):
    fig, axes = plt.subplots(1, len(ids), figsize=(5 * len(ids), 5))
    gt = U.build_gt(targets, ids)
    for ax, i in zip(np.ravel(axes), ids):
        U.show_predictions(i, gt, gt.iloc[0:0].assign(score=0), PNG_DIR, ax=ax,
                           title=f'{i[:10]} p_abn={oof.p_abn[i]:.3f}')
    fig.suptitle(title); plt.tight_layout(); plt.show()

fn = oof[y.abnormal == 1].p_abn.nsmallest(4).index      # abnormal, scored as normal
fp = oof[y.abnormal == 0].p_abn.nlargest(4).index       # normal, scored as abnormal
show_ids(fn, 'False negatives: abnormal images with the lowest p_abn (GT boxes in green)')
show_ids(fp, 'False positives: normal images with the highest p_abn')
print('Classes present in the false negatives:')
display(targets[targets.image_id.isin(oof[y.abnormal == 1].p_abn.nsmallest(100).index)]
        .class_id.map(dict(enumerate(U.CLASSES))).value_counts().to_frame('boxes in 100 lowest-scored abnormal images'))